**Read Bronze tables, clean the data, and write Silver tables.**

In [0]:
from pyspark.sql.functions import col, when, trim, lit, coalesce, to_date
from pyspark.sql.types import IntegerType, StringType, DoubleType

DELTA_VOLUME_PATH = "/Volumes/bc_transit_ws/gtfs/delta_volume/"
BRONZE_BASE_PATH = f"{DELTA_VOLUME_PATH}bronze/"
SILVER_BASE_PATH = f"{DELTA_VOLUME_PATH}silver/"

In [0]:
# List of bronze tables to access
bronze_tables = [
    "raw_agency",
    "raw_routes",
    "raw_trips",
    "raw_calendar",
    "raw_stops",
    "raw_stop_times"
]

# Dicitonary to store Bronze Dataframes
bronze_dataframes = {}

# Loop through bronze tables and create DataFrames
for table in bronze_tables:
    path = BRONZE_BASE_PATH + table
    
    # read the delta table
    df = spark.read.format("delta").load(path)

    # add the dataframe to the dictionary
    bronze_dataframes[table] = df
    print(f"Read bronze table: {table}")
    

**Standardize and Write Silver Tables**

In [0]:
# Agency table

df_raw_agency = bronze_dataframes["raw_agency"]

df_silver_agency = df_raw_agency.select(
      col("agency_id").cast(StringType()).alias("agency_id"),
      col("agency_name").cast(StringType()).alias("agency_name"),
      col("agency_url").cast(StringType()).alias("agency_url"),
      col("agency_timezone").cast(StringType())
).distinct()

# "/Volumes/bc_transit_ws/gtfs/delta_volume/silver/agency"
silver_agency_path = f"{SILVER_BASE_PATH}agency/"

( df_silver_agency.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .save(silver_agency_path)
)

print(f"Wrote silver table: agency to {silver_agency_path}")


In [0]:
# Routes table
df_raw_routes = bronze_dataframes["raw_routes"]

# 1 (Train), 2 (WCE), 4 (SeaBus), 715 (HandyDART)
NON_BUS_TYPES = [1, 2, 4, 715]

df_silver_routes = df_raw_routes.select(
        col("route_id").alias("id").cast(StringType()).alias("route_id"),
        col("agency_id").cast(StringType()).alias("agency_id"),

        coalesce(
              trim(col("route_short_name")),
              when(
                  col("route_type").isin(NON_BUS_TYPES),
                  col("route_long_name") # Use the long name for trains, SeaBus, WCE, etc.
              ).otherwise(
                  col("route_long_name") # for all other cases
              )
        ).alias("route_short_name_display"),

        col("route_long_name").cast(StringType()).alias("route_long_name"),
        col("route_type").cast(IntegerType()).alias("route_type"),
        
        coalesce(col("route_color").cast(StringType()), lit("FFFFFF")).alias("route_color"),
        coalesce(col("route_text_color").cast(StringType()), lit("000000")).alias("route_text_color").alias("route_text_color")

        
).filter(col("route_id").isNotNull()).distinct()

# "/Volumes/bc_transit_ws/gtfs/delta_volume/silver/routes"
silver_routes_path = f"{SILVER_BASE_PATH}routes/"

( df_silver_routes.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .save(silver_routes_path)
)   
print(f"Wrote silver table: routes to {silver_routes_path}")

In [0]:
# Trips table
'''
route_id	service_id	trip_id	trip_headsign	trip_short_name	direction_id	block_id	shape_id	wheelchair_accessible	bikes_allowed
'''

df_raw_trips = bronze_dataframes["raw_trips"]
HANDYDART_ROUTE_ID = "HD"

df_silver_trips = df_raw_trips.select(
        col("route_id").cast(StringType()).alias("route_id"),
        col("service_id").cast(IntegerType()).alias("service_id"),
        col("trip_id").cast(IntegerType()).alias("trip_id"),

        coalesce(
            trim(col("trip_headsign")),
            when(
              col("route_id") == lit(HANDYDART_ROUTE_ID),
              lit("HandyDART")
            ).otherwise(
              lit("Unknown Destination")
            )
        ).alias("trip_headsign"),

        col("direction_id").cast(IntegerType()).alias("direction_id"),
        col("block_id").cast(IntegerType()).alias("block_id"),
        col("shape_id").cast(IntegerType()).alias("shape_id"),
        col("wheelchair_accessible").cast(IntegerType()).alias("wheelchair_acessible"),
        col("bikes_allowed").cast(IntegerType()).alias("bikes_allowed")

).filter(col("trip_id").isNotNull())

silver_trips_path = f"{SILVER_BASE_PATH}trips/"

( df_silver_trips.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .save(silver_trips_path)
)

print(f"Wrote silver table: trips to {silver_trips_path}")

In [0]:
# Calendar table

df_raw_calendar = bronze_dataframes["raw_calendar"]

df_silver_calendar = df_raw_calendar.select(
        col("service_id").cast(StringType()).alias("service_id"),
        col("monday").cast(IntegerType()),
        col("tuesday").cast(IntegerType()),
        col("wednesday").cast(IntegerType()),
        col("thursday").cast(IntegerType()),
        col("friday").cast(IntegerType()),
        col("saturday").cast(IntegerType()),
        col("sunday").cast(IntegerType()),
        to_date(col("start_date"), "yyyyMMdd").alias("start_date"),
        to_date(col("end_date"), "yyyyMMdd").alias("end_date")

).filter(col("service_id").isNotNull()).distinct()

silver_calendar_path = f"{SILVER_BASE_PATH}calendar/"

(df_silver_calendar.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .save(silver_calendar_path)
)
print(f"Wrote silver table: trips to {silver_calendar_path}")



In [0]:
# Stops table


df_raw_stops = bronze_dataframes['raw_stops']

df_silver_stops = (
    df_raw_stops
    .select(
        col("stop_id").cast(StringType()).alias("stop_id"), # Primary Key
        col("stop_code").cast(StringType()).alias("stop_code"),
        col("zone_id").cast(StringType()).alias("zone_id"),
        col("stop_lat").cast(DoubleType()).alias("stop_lat"),
        col("stop_lon").cast(DoubleType()).alias("stop_lon"),
        col("stop_name").cast(StringType()).alias("stop_name_detail"), 

        # If parent_station is empty or just whitespace, convert it to NULL
        when(
            trim(col("parent_station")) == lit(""),
            lit(None).cast(StringType())
        ).otherwise(
            col("parent_station").cast(StringType())
        ).alias("parent_station"), 

        col("location_type").cast(IntegerType()).alias("location_type"),
        col("wheelchair_boarding").cast(IntegerType()).alias("wheelchair_boarding")
    )
    .filter(col("stop_id").isNotNull())
    .distinct()
)

silver_stops_path = f"{SILVER_BASE_PATH}stops/"

(df_silver_stops.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_stops_path)
)

print(f"Wrote silver table: stops to {silver_stops_path}")


In [0]:
# Stop Times Table


df_raw_stop_times = bronze_dataframes['raw_stop_times']

df_silver_stop_times = (
    df_raw_stop_times
    .select(
        col("trip_id").cast(StringType()).alias("trip_id"), 
        col("stop_id").cast(StringType()).alias("stop_id"), 
        col("stop_sequence").cast(IntegerType()).alias("stop_sequence"),
        col("arrival_time").cast(StringType()).alias("arrival_time"),
        col("departure_time").cast(StringType()).alias("departure_time"),
  
        # Use coalesce to replace null/empty shape_dist_traveled with 0.0
        coalesce(col("shape_dist_traveled").cast(DoubleType()), lit(0.0)).alias("shape_dist_traveled"),
        col("timepoint").cast(IntegerType()).alias("timepoint")
    )
    .filter(col("trip_id").isNotNull())
    .filter(col("stop_id").isNotNull())
    .distinct()
)

silver_stop_times_path = f"{SILVER_BASE_PATH}stop_times/"

(df_silver_stop_times.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_stop_times_path)
)

print(f"Wrote silver table stop times to: {silver_stop_times_path}")

In [0]:
# Unity Catalog Managed Table Registration


CATALOG_NAME = "bc_transit_ws"
SCHEMA_NAME = "silver"


spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")


# List of Silver tables 
silver_tables_to_register = ["agency", "routes", "trips", "calendar", "stops", "stop_times"]

for table_name in silver_tables_to_register:
    
    # Path where data was written earlier in the notebook
    path = f"{SILVER_BASE_PATH}{table_name}/" 
    full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{table_name}"
    
    # Reading the data from the volume path where it;s saved before
    df_to_register = spark.read.format("delta").load(path)
    
    # saveAsTable 
    (df_to_register.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name) 
    )
    
    print(f"Registered Managed Table: {full_table_name}")
    
print("\nSilver Layer Processing and Managed Registration Complete.")